<a href="https://colab.research.google.com/github/ad1tya-io/Heart_Disease_Prediction_System/blob/main/Heart_Disease_Prediction_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Heart Disease Prediction System**

In [1]:
# Adding necessary libraries
import numpy as np
import pandas as pd

# To visualize the data
import matplotlib.pyplot as plt
import seaborn as sns

First, we would need to get a glimplse of the **Dataset**.

In [2]:
# Loading the dataset
data_url = "https://raw.githubusercontent.com/ad1tya-io/Heart_Disease_Prediction_System/refs/heads/main/dataset/heart_disease_uci.csv"


df = pd.read_csv(data_url)

print(f"Dataset loaded successfully! Shape: {df.shape}")
df.head()

Dataset loaded successfully! Shape: (920, 16)


,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


In [4]:
# Shape of dataset
print("Dataset Dimensions: ", df.shape)

# Column names
print("\n Column Names \n", df.columns.tolist())

Dataset Dimensions:  (920, 16)

 Column Names 
 ['id', 'age', 'sex', 'dataset', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalch', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num']


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 920 entries, 0 to 919
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        920 non-null    int64  
 1   age       920 non-null    int64  
 2   sex       920 non-null    object 
 3   dataset   920 non-null    object 
 4   cp        920 non-null    object 
 5   trestbps  861 non-null    float64
 6   chol      890 non-null    float64
 7   fbs       830 non-null    object 
 8   restecg   918 non-null    object 
 9   thalch    865 non-null    float64
 10  exang     865 non-null    object 
 11  oldpeak   858 non-null    float64
 12  slope     611 non-null    object 
 13  ca        309 non-null    float64
 14  thal      434 non-null    object 
 15  num       920 non-null    int64  
dtypes: float64(5), int64(3), object(8)
memory usage: 115.1+ KB


Now we would do simple ***Data Preprocessing***.

In [8]:
# Identitfying Missing values
print("\n Missing Values \n", df.isnull().sum())

# Identifying values that have zeroes
print("\n Zero values\n", (df == 0).sum())



 Missing Values 
 id            0
age           0
sex           0
dataset       0
cp            0
trestbps     59
chol         30
fbs          90
restecg       2
thalch       55
exang        55
oldpeak      62
slope       309
ca          611
thal        486
num           0
dtype: int64

 Zero values
 id            0
age           0
sex           0
dataset       0
cp            0
trestbps      1
chol        172
fbs         692
restecg       0
thalch        0
exang       528
oldpeak     370
slope         0
ca          181
thal          0
num         411
dtype: int64


We have missing values over 10 columns:


*   trestbps
*   chol
*   fbs
*   restecg
*   thalch
*   exang
*   oldpeak
*   slope
*   ca
*   thal  
We would need to resolve these missing values.

1. **trestbps** - this is the resting blood pressure. It is important data feature and needs to be handled carefully for our model's accuracy and precision.
- we could use the mean of the *trestbps* to fill the missing values, but the missing data is around 6% of the dataset, so it would not be appropriately accurate.

- we will use the mean of the selective values of *trestbps* which has some correlated similarities through sex and age.

In [10]:
# We create age groups to ensure robust group sizes
age_bins = [0, 29, 39, 49, 59, 69, 79, 100]
age_labels = ['<30', '30s', '40s', '50s', '60s', '70s', '80+']

# We create a new column which we will drop later.
df['age_group'] = pd.cut(df['age'], bins=age_bins, labels=age_labels)

# We would also handle potential 0 values
df['trestbps'] = df['trestbps'].replace(0, np.nan)

# we calculate group means and fill the missing values
df['trestbps'] = df.groupby(['age_group', 'sex'])['trestbps'].transform(
    lambda x: x.fillna(x.mean())
)

# we drop the column in the end
df = df.drop(columns=['age_group'])

# checking the missing and zero values again
total_missing = (df['trestbps'] == 0).sum() + df['trestbps'].isna().sum()
print(total_missing)


0


/tmp/ipykernel_3436/1691400216.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df['trestbps'] = df.groupby(['age_group', 'sex'])['trestbps'].transform(
